# Baseline: no_aerosols_zero_wind — 3D vs 1D

Diagnostics for the **no aerosols, zero wind** baseline experiment.

| Section | Content |
|---------|---------|
| **1. LWP** | 1D vs 3D timeseries and 3D − 1D difference + integral |
| **2. Surface energy balance** | Domain-mean SEB, residual, and cloud-conditioned fluxes |
| **3. Sub-cloud turbulent flux profiles** | Cloud-root vs cloud-free absolute H/LE profiles + decomposition |


In [ ]:
import sys
import os
import pickle
import warnings
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from pathlib import Path
from matplotlib.lines import Line2D

_HERE = Path.cwd()
sys.path.insert(0, str(_HERE))
sys.path.insert(0, str(_HERE / 'cloud_roots'))

from cass_analysis import (
    load_stats, load_stats_ensemble,
    RunSet,
    conditioned_means_ensemble,
    lwp_integral, _to_plottime,
    seb_residual,
    plot_seb_timeseries, plot_seb_residual, plot_flux_conditioned,
    sim_time_to_lst, lst_window_mask,
    FLUX_COLORS, RT_STYLE, RT_LABEL,
    rho, cp, Lv, LST_OFFSET,
)
from catalog import make_runset, CASS_ROOT

In [ ]:
# ── experiment ────────────────────────────────────────────────────────────────
EXPT_KEY       = 'no_aerosols_zero_wind'
N_REPS         = 4
rs             = make_runset(EXPT_KEY, n_reps=N_REPS)
print(rs)

# ── cloud-root profile / composite config ─────────────────────────────────────
COMPOSITE_ROOT = str(CASS_ROOT / 'analysis' / 'cloud_root_composite')
LST_MIN        = 10.5
LST_MAX        = 17.5
ORIENT         = 'xz'
SAVE_PDF       = False

# ── CASES for cloud-root profiles / composites ────────────────────────────────
CASES = [
    dict(key='1D', rt='2stream',   comp_expt=EXPT_KEY, comp_rt='2stream'),
    dict(key='3D', rt='raytracer', comp_expt=EXPT_KEY, comp_rt='raytracer'),
]

In [ ]:
# ── LWP loading ───────────────────────────────────────────────────────────────
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    dirs_2s = rs.dirs.get('2stream', [])
    dirs_rt = rs.dirs.get('raytracer', [])

reps_2s  = [load_stats(d) for d in dirs_2s]
reps_rt  = [load_stats(d) for d in dirs_rt]
n_paired = min(len(reps_2s), len(reps_rt))
nt       = min(*(s['qlqi_path'].sizes['time'] for s in reps_2s + reps_rt))

lwp_1d_reps = np.stack([s['qlqi_path'].values[:nt] * 1e3 for s in reps_2s[:n_paired]])
lwp_3d_reps = np.stack([s['qlqi_path'].values[:nt] * 1e3 for s in reps_rt[:n_paired]])
diff_reps   = lwp_3d_reps - lwp_1d_reps

lwp_1d     = lwp_1d_reps.mean(axis=0)
lwp_3d     = lwp_3d_reps.mean(axis=0)
lwp_1d_std = lwp_1d_reps.std(axis=0, ddof=0)
lwp_3d_std = lwp_3d_reps.std(axis=0, ddof=0)
diff       = diff_reps.mean(axis=0)
diff_std   = diff_reps.std(axis=0, ddof=0)

t_local = reps_2s[0]['t_local'].values[:nt]
t_sec   = reps_2s[0]['t_sec'].values[:nt]
dt_s    = float(t_sec[1] - t_sec[0]) if len(t_sec) > 1 else 60.0

cloudy_mask  = (lwp_1d > 0.1) | (lwp_3d > 0.1)
int_diff     = lwp_integral(diff, dt_s, cloudy_mask=cloudy_mask)
int_1d       = lwp_integral(lwp_1d, dt_s, cloudy_mask=cloudy_mask)
norm_int     = int_diff / int_1d * 100 if abs(int_1d) > 1e-6 else np.nan
norm_int_std = float(np.std(
    [lwp_integral(diff_reps[r], dt_s, cloudy_mask=cloudy_mask) / int_1d * 100
     for r in range(n_paired)], ddof=0)) if abs(int_1d) > 1e-6 else np.nan

print(f'LWP loaded:  n_paired={n_paired},  nt={nt}')
print(f'Normalised integral: {norm_int:+.2f} \u00b1 {norm_int_std:.2f} %')

## 1. LWP


In [ ]:
# ── Figure 1: LWP 1D vs 3D (top) + 3D − 1D difference (bottom) ───────────────
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
t_num = _to_plottime(t_local)

ax = axes[0]
ax.plot(t_num, lwp_1d, color='C0', lw=1.5, label='1D (2stream)')
ax.fill_between(t_num, lwp_1d - lwp_1d_std, lwp_1d + lwp_1d_std, color='C0', alpha=0.3)
ax.plot(t_num, lwp_3d, color='C1', lw=1.5, ls='--', label='3D (raytracer)')
ax.fill_between(t_num, lwp_3d - lwp_3d_std, lwp_3d + lwp_3d_std, color='C1', alpha=0.3)
ax.set_ylabel(r'LWP  (g m$^{-2}$)')
ax.axhline(0, color='gray', lw=0.5)
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(t_num, diff, color='k', lw=1.8)
ax.fill_between(t_num, diff - diff_std, diff + diff_std, color='k', alpha=0.2)
ax.axhline(0, color='gray', lw=0.7, ls='--')
ax.set_ylabel(r'3D $-$ 1D LWP  (g m$^{-2}$)')
ax.text(0.02, 0.95,
        fr'$\int$(3D $-$ 1D) / $\int$(1D) = {norm_int:+.1f} $\pm$ {norm_int_std:.1f} %',
        transform=ax.transAxes, fontsize=9, va='top')
ax.xaxis_date()
fig.autofmt_xdate()

fig.suptitle(
    fr'{EXPT_KEY}: domain-mean LWP  (ensemble mean $\pm1\sigma$, $n$={n_paired})',
    fontsize=11,
)
plt.tight_layout()
if SAVE_PDF:
    plt.savefig('lwp_baseline.pdf', bbox_inches='tight')
    print('Saved lwp_baseline.pdf')
plt.show()

### Comparison: no_aerosols_zero_wind vs sw_scale

`sw_scale` runs the raytracer with `swscalesfc_to_2str=true`, which rescales the 3D
surface SW each radiation step so its domain mean matches the 2stream mean. The
remaining 3D−1D LWP difference is therefore driven purely by spatial SW heterogeneity
(shadow displacement), with no contribution from mean SW bias.

In [ ]:
# ── Load sw_scale experiment (raytracer only, swscalesfc_to_2str=true) ──────
sw_scale_root = CASS_ROOT / 'experiments' / 'sw_scale' / 'raytracer'
sw_dirs = sorted([d for d in sw_scale_root.glob('rep_*') if d.is_dir()])

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    reps_sw = [load_stats(d) for d in sw_dirs]

n_sw = min(len(reps_2s), len(reps_sw))
nt_sw = min(*(s['qlqi_path'].sizes['time'] for s in reps_2s + reps_sw))

lwp_1d_sw_reps = np.stack([s['qlqi_path'].values[:nt_sw] * 1e3 for s in reps_2s[:n_sw]])
lwp_sw_reps    = np.stack([s['qlqi_path'].values[:nt_sw] * 1e3 for s in reps_sw[:n_sw]])
diff_sw_reps   = lwp_sw_reps - lwp_1d_sw_reps

diff_sw     = diff_sw_reps.mean(axis=0)
diff_sw_std = diff_sw_reps.std(axis=0, ddof=0)

# Normalized integral for sw_scale
cloudy_mask_sw = (lwp_1d_sw_reps.mean(0) > 0.1) | (lwp_sw_reps.mean(0) > 0.1)
int_diff_sw = lwp_integral(diff_sw, dt_s, cloudy_mask=cloudy_mask_sw)
int_1d_sw   = lwp_integral(lwp_1d_sw_reps.mean(0), dt_s, cloudy_mask=cloudy_mask_sw)
norm_sw     = int_diff_sw / int_1d_sw * 100 if abs(int_1d_sw) > 1e-6 else np.nan

# Truncate base diff to match length
nt_min = min(nt, nt_sw)
diff_base     = diff[:nt_min]
diff_base_std = diff_std[:nt_min]
diff_sw_p     = diff_sw[:nt_min]
diff_sw_p_std = diff_sw_std[:nt_min]
t_num_sw = _to_plottime(reps_2s[0]['t_local'].values[:nt_min])

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t_num_sw, diff_base, color='C0', lw=1.8,
        label=fr'no_aerosols_zero_wind (full 3D $-$ 1D)') # $\int / \int_{{1D}}$={norm_int:+.1f}%
ax.fill_between(t_num_sw, diff_base - diff_base_std, diff_base + diff_base_std,
                color='C0', alpha=0.2)
ax.plot(t_num_sw, diff_sw_p, color='C3', lw=1.8, ls='--',
        label=fr'sw_scale (mean-rescaled 3D $-$ 1D)') # , $\int / \int_{{1D}}$={norm_sw:+.1f}%
ax.fill_between(t_num_sw, diff_sw_p - diff_sw_p_std, diff_sw_p + diff_sw_p_std,
                color='C3', alpha=0.2)
ax.axhline(0, color='gray', lw=0.7, ls=':')
ax.set_ylabel(r'3D $-$ 1D LWP  (g m$^{-2}$)')
ax.xaxis_date()
ax.legend(fontsize=9, loc='upper left')
ax.set_title('Mean bias vs heterogeneity contribution to 3D$-$1D LWP difference')
fig.autofmt_xdate()
fig.tight_layout()

print(f'no_aerosols_zero_wind: int (3D-1D) / int 1D = {norm_int:+.2f}%')
print(f'sw_scale:              int (3D-1D) / int 1D = {norm_sw:+.2f}%')
print(f'Heterogeneity-only fraction: {norm_sw/norm_int*100:+.0f}% of total')


## 2. Surface energy balance


In [ ]:
# ── SEB: load ensemble stats ──────────────────────────────────────────────────
s2s_mean, s2s_std = load_stats_ensemble(rs.dirs['2stream'])
srt_mean, srt_std = load_stats_ensemble(rs.dirs['raytracer'])

print(f"2stream:   {len(rs.dirs['2stream'])} rep(s),  "
      f"t = {s2s_mean['t_local'].values[0]} → {s2s_mean['t_local'].values[-1]}")
print(f"raytracer: {len(rs.dirs['raytracer'])} rep(s),  "
      f"t = {srt_mean['t_local'].values[0]} → {srt_mean['t_local'].values[-1]}")

# ── SEB: load conditioned means (pickle cache) ────────────────────────────────
_SEB_CACHE = CASS_ROOT / 'analysis' / 'seb_cache'
_SEB_CACHE.mkdir(parents=True, exist_ok=True)

def _compute_cond(result):
    mean_d, std_d = result
    for d in (mean_d, std_d):
        for kind in d:
            d[kind] = {var: da.compute() for var, da in d[kind].items()}
    return mean_d, std_d

def _load_or_compute_conditioned(rep_dirs, cache_name, variables, force=False):
    cache_file = _SEB_CACHE / f'{cache_name}.pkl'
    if not force and cache_file.exists():
        with open(cache_file, 'rb') as fh:
            cached = pickle.load(fh)
        # Rebuild if cache predates 'domain' key
        if 'domain' in cached[0]:
            print(f'  [cache hit] {cache_file.name}')
            return cached
        print(f'  [stale cache — missing domain key] {cache_file.name}')
    print(f'  [computing] {cache_name} …')
    result = _compute_cond(conditioned_means_ensemble(rep_dirs, variables=variables))
    with open(cache_file, 'wb') as fh:
        pickle.dump(result, fh)
    print(f'  [cached]    {cache_file.name}')
    return result

_slug = f'experiments_{EXPT_KEY}'
cond_2s_mean, cond_2s_std = _load_or_compute_conditioned(
    rs.dirs['2stream'],
    f'{_slug}__2stream',
    variables=['thl_fluxbot', 'qt_fluxbot', 'sw_flux_dn'],
)
cond_rt_mean, cond_rt_std = _load_or_compute_conditioned(
    rs.dirs['raytracer'],
    f'{_slug}__raytracer',
    variables=['thl_fluxbot', 'qt_fluxbot', 'sw_flux_sfc_rt', 'sw_flux_dn'],
)
print('Done.')


In [ ]:
# ── Figure 2a: domain-mean SEB + residual ────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax = axes[0]
for rt, sm, ss in [('2stream', s2s_mean, s2s_std), ('raytracer', srt_mean, srt_std)]:
    plot_seb_timeseries(ax, sm, ss, rt_type=rt)
flux_handles = [Line2D([0],[0], color=c, ls='-', label=f) for f, c in FLUX_COLORS.items()]
rt_handles   = [Line2D([0],[0], color='gray', ls=sty['ls'], label=RT_LABEL[rt])
                for rt, sty in RT_STYLE.items()]
ax.legend(handles=flux_handles + rt_handles, fontsize=7, ncol=2)
ax.set_title(f'{EXPT_KEY}  —  Surface energy balance')

ax = axes[1]
for rt, sm in [('2stream', s2s_mean), ('raytracer', srt_mean)]:
    plot_seb_residual(ax, sm, rt_type=rt)
for rt, sm in [('2stream', s2s_mean), ('raytracer', srt_mean)]:
    resid   = seb_residual(sm)
    daytime = sm['Rnet'] > 50
    print(f'{rt}  peak|resid|={np.abs(resid).max():.1f}  '
          f'daytime_mean|resid|={np.abs(resid[daytime]).mean():.1f} W m⁻²')
ax.legend(fontsize=8)

plt.tight_layout()
if SAVE_PDF:
    plt.savefig('seb_baseline.pdf', bbox_inches='tight')
    print('Saved seb_baseline.pdf')
plt.show()


In [ ]:
# ── Figure 2b: SEB conditioned on cloud mask ─────────────────────────────────
# Order: SW↓, H, LE.  Each panel includes domain mean from same xy fields.
fig, axes = plt.subplots(3, 2, figsize=(13, 10), sharex=True)

H_scale  = rho * cp
LE_scale = rho * Lv

for col, (rt, cm, cs) in enumerate([
    ('2stream',   cond_2s_mean, cond_2s_std),
    ('raytracer', cond_rt_mean, cond_rt_std),
]):
    t_local_c = cm['shaded']['thl_fluxbot'].time.values

    # Row 0: SW↓
    sw_key = 'sw_flux_sfc_rt' if 'sw_flux_sfc_rt' in cm['shaded'] else 'sw_flux_dn'
    plot_flux_conditioned(
        axes[0, col], t_local_c,
        cm['shaded'][sw_key], cm['unshaded'][sw_key],
        cs['shaded'].get(sw_key), cs['unshaded'].get(sw_key),
        rt_type=rt, ylabel='SW↓  (W m⁻²)', scale=1.0,
    )
    dm_sw = cm['domain'][sw_key]
    axes[0, col].plot(_to_plottime(dm_sw.time.values), dm_sw.values,
                      color='k', ls='--', lw=1.2, label='domain mean')
    axes[0, col].set_title(f'{RT_LABEL[rt]}:  Surface SW irradiance')

    # Row 1: H
    plot_flux_conditioned(
        axes[1, col], t_local_c,
        cm['shaded']['thl_fluxbot'], cm['unshaded']['thl_fluxbot'],
        cs['shaded'].get('thl_fluxbot'), cs['unshaded'].get('thl_fluxbot'),
        rt_type=rt, ylabel='H  (W m⁻²)', scale=H_scale,
    )
    dm_h = cm['domain']['thl_fluxbot']
    axes[1, col].plot(_to_plottime(dm_h.time.values), dm_h.values * H_scale,
                      color='k', ls='--', lw=1.2, label='domain mean')
    axes[1, col].set_title(f'{RT_LABEL[rt]}:  Sensible heat flux')

    # Row 2: LE
    plot_flux_conditioned(
        axes[2, col], t_local_c,
        cm['shaded']['qt_fluxbot'], cm['unshaded']['qt_fluxbot'],
        cs['shaded'].get('qt_fluxbot'), cs['unshaded'].get('qt_fluxbot'),
        rt_type=rt, ylabel='LE  (W m⁻²)', scale=LE_scale,
    )
    dm_le = cm['domain']['qt_fluxbot']
    axes[2, col].plot(_to_plottime(dm_le.time.values), dm_le.values * LE_scale,
                      color='k', ls='--', lw=1.2, label='domain mean')
    axes[2, col].set_title(f'{RT_LABEL[rt]}:  Latent heat flux')

for ax in axes.flat:
    ax.legend(fontsize=7)

plt.suptitle(f'{EXPT_KEY}: SEB conditioned on cloud mask', y=1.01)
plt.tight_layout()
if SAVE_PDF:
    plt.savefig('seb_conditioned_baseline.pdf', bbox_inches='tight')
    print('Saved seb_conditioned_baseline.pdf')
plt.show()


In [ ]:
# ── Figure 2d: mean profiles by SZA window (1D vs 3D) ────────────────────────
# Mirrors the θ_z windows used for the circulation composites.
# Panels: θ_l, q_t, u, v, TKE (e), q_l, w'θ_v' (buoyancy flux)

import pandas as pd
s2s_mean, s2s_std = load_stats_ensemble(rs.dirs['2stream'])
srt_mean, srt_std = load_stats_ensemble(rs.dirs['raytracer'])

def _lst_hours(t_local):
    """Convert pandas DatetimeIndex / array to fractional LST hours."""
    t = pd.DatetimeIndex(t_local)
    return t.hour + t.minute / 60.0 + t.second / 3600.0

PROFILE_SPECS = [
    ('tke', 'z',  r'TKE  (m$^2$ s$^{-2}$)', 1.0),
    ('ql',  'z',  r'$q_l$  (g kg$^{-1}$)',  1e3),
    ('thv_w','zh', r"$\overline{w'\theta_v'}$  (K m s$^{-1}$)", 1.0),
]

from sw_surface_composite import WINDOWS as SZA_WINDOWS, WINDOW_LABELS as SZA_LABELS
n_win  = len(SZA_WINDOWS)
n_vars = len(PROFILE_SPECS)
Z_MAX  = 4000  # m — plot up to this height

fig, axes = plt.subplots(1, n_vars, figsize=(3.2 * n_vars, 6), sharey=True)

win_colors = ['#4292c6', '#fd8d3c', '#8B0000']  # same as SW composite windows
if n_win > len(win_colors):
    from matplotlib.cm import get_cmap
    win_colors = [get_cmap('viridis')(i / (n_win - 1)) for i in range(n_win)]

for rt, sm, ls_style, rt_lbl in [
    ('2stream',   s2s_mean, '-',  '1D'),
    ('raytracer', srt_mean, '--', '3D'),
]:
    lst_h = _lst_hours(sm['t_local'].values)
    for wi, (lst_lo, lst_hi) in enumerate(SZA_WINDOWS):
        mask = (lst_h >= lst_lo) & (lst_h <= lst_hi)
        if mask.sum() == 0:
            continue
        for col, (var, zgrid, xlabel, scale) in enumerate(PROFILE_SPECS):
            ax = axes[col]
            z  = sm[zgrid].values
            prof = sm[var].values[mask].mean(axis=0) * scale
            z_mask = z <= Z_MAX
            label = f'{rt_lbl} {SZA_LABELS[wi][:11]}' if col == 0 else None
            ax.plot(prof[z_mask], z[z_mask], color=win_colors[wi],
                    ls=ls_style, lw=1.5, label=label)
            ax.set_xlabel(xlabel, fontsize=9)

axes[0].set_ylabel('Height  (m)')
axes[0].set_ylim(0, Z_MAX)
# Build legend: line style = RT, colour = window
from matplotlib.lines import Line2D
rt_handles = [
    Line2D([0],[0], color='gray', ls='-',  lw=1.5, label='1D (2stream)'),
    Line2D([0],[0], color='gray', ls='--', lw=1.5, label='3D (raytracer)'),
]
win_handles = [
    Line2D([0],[0], color=win_colors[wi], ls='-', lw=3,
           label=SZA_LABELS[wi].split('(')[0].strip())
    for wi in range(n_win)
]
axes[0].legend(handles=rt_handles + win_handles, fontsize=6.5,
               loc='upper left', frameon=True)

plt.tight_layout()
if SAVE_PDF:
    plt.savefig('mean_profiles_windowed.pdf', bbox_inches='tight')
    print('Saved mean_profiles_windowed.pdf')
plt.show()


## q_l time-height cross-section

3D (raytracer), 1D (2stream), and 3D−1D domain-mean q_l, averaged over 4 reps.
Uses stats profiles loaded earlier for the SEB section.


In [ ]:
# ── q_l time-height cross-sections: 1D, 3D, difference ──────────────────────
z_max = 5000

z     = srt_mean['z'].values
zmsk  = z <= z_max
z_ax  = z[zmsk]

t1 = s2s_mean['t_sec'].values
t3 = srt_mean['t_sec'].values
nt = min(len(t1), len(t3))
ql_1d = s2s_mean['ql'].isel(time=slice(None, nt)).values[:, zmsk] * 1e3  # g/kg
ql_3d = srt_mean['ql'].isel(time=slice(None, nt)).values[:, zmsk] * 1e3
ql_diff = ql_3d - ql_1d
t_h = t3[:nt] / 3600.0 + LST_OFFSET

# Shared colour scales
_pos  = np.concatenate([ql_3d[ql_3d > 0].ravel(), ql_1d[ql_1d > 0].ravel()])
vmax_abs  = max(float(np.nanpercentile(_pos, 99)), 0.01) if _pos.size else 0.01
_dif_flat = np.abs(ql_diff).ravel()
vmax_diff = max(float(np.nanpercentile(_dif_flat[np.isfinite(_dif_flat)], 99)), 0.01)

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True, sharey=True)

pcm_abs  = axes[0].pcolormesh(t_h, z_ax, ql_3d.T, cmap='Blues',
                              vmin=0, vmax=vmax_abs)
axes[1].pcolormesh(t_h, z_ax, ql_1d.T, cmap='Blues',
                   vmin=0, vmax=vmax_abs)
pcm_diff = axes[2].pcolormesh(t_h, z_ax, ql_diff.T, cmap='RdBu_r',
                               vmin=-vmax_diff, vmax=vmax_diff)

for ax, label in zip(axes, ['3D (raytracer)', '1D (2stream)', '3D − 1D']):
    ax.set_ylabel(f'z (m)\n{label}')
axes[2].set_xlabel('LST  (h)')

fig.colorbar(pcm_abs,  ax=axes[:2].tolist(), location='right', shrink=0.7,
             label=r'$q_l$  (g kg$^{-1}$)')
fig.colorbar(pcm_diff, ax=axes[2],           location='right', shrink=0.7,
             label=r'$\Delta q_l$  (g kg$^{-1}$)')

fig.suptitle(f'{EXPT_KEY}: q_l time-height (ensemble mean)')
if SAVE_PDF:
    plt.savefig(f'ql_timeh_{EXPT_KEY}.pdf', bbox_inches='tight')
plt.show()


## 3. Sub-cloud turbulent flux profiles

Reads the cloud-root cache built by `cloud_roots/free_vs_nudge_cloud_roots.ipynb` (Section A).  
Run that notebook first if the cache does not exist.


In [ ]:
# ── Derived constants ─────────────────────────────────────────────────────────
RHO_CP = rho * cp
RHO_LV = rho * Lv * 1e-3


def load_absolute_profiles(case, n_reps=N_REPS, lst_min=LST_MIN, lst_max=LST_MAX):
    """Load cached raw flux profiles and convert to H / LE in W m⁻².
    Cache must exist — run Section A of cloud_roots/free_vs_nudge_cloud_roots.ipynb first.
    """
    H_dom_reps, H_root_reps, H_free_reps   = [], [], []
    LE_dom_reps, LE_root_reps, LE_free_reps = [], [], []
    zeta = None

    for i in range(1, n_reps + 1):
        cache_path = (Path(COMPOSITE_ROOT) / case['comp_expt'] / case['comp_rt']
                      / f'rep_{i:02d}' / 'raw_flux_profile_cache.nc')
        if not cache_path.exists():
            print(f'  SKIP {case["key"]} rep_{i:02d}: cache not found')
            continue
        ds_c = xr.open_dataset(str(cache_path))
        if 'f_free_thl' not in ds_c or 't_hours' not in ds_c:
            ds_c.close()
            print(f'  SKIP {case["key"]} rep_{i:02d}: cache outdated')
            continue

        zeta   = ds_c['zeta'].values
        t_hrs  = ds_c['t_hours'].values
        mask_t = (t_hrs >= lst_min) & (t_hrs <= lst_max)

        if mask_t.sum() == 0:
            ds_c.close()
            continue

        H_dom_reps.append( np.nanmean(ds_c['f_domain_thl'].values[mask_t] * RHO_CP, axis=0))
        H_root_reps.append(np.nanmean(ds_c['f_cloud_thl'].values[mask_t]  * RHO_CP, axis=0))
        H_free_reps.append(np.nanmean(ds_c['f_free_thl'].values[mask_t]   * RHO_CP, axis=0))
        LE_dom_reps.append( np.nanmean(ds_c['f_domain_qv'].values[mask_t] * RHO_LV, axis=0))
        LE_root_reps.append(np.nanmean(ds_c['f_cloud_qv'].values[mask_t]  * RHO_LV, axis=0))
        LE_free_reps.append(np.nanmean(ds_c['f_free_qv'].values[mask_t]   * RHO_LV, axis=0))
        ds_c.close()

    def _s(lst): return np.stack(lst) if lst else np.empty((0,))
    return dict(zeta=zeta, n_reps=len(H_dom_reps),
                H_dom=_s(H_dom_reps),   H_root=_s(H_root_reps),  H_free=_s(H_free_reps),
                LE_dom=_s(LE_dom_reps), LE_root=_s(LE_root_reps), LE_free=_s(LE_free_reps))


def load_cloud_frac(case, n_reps=N_REPS, lst_min=LST_MIN, lst_max=LST_MAX):
    """Time-mean cloud-root area fraction per rep."""
    fracs = []
    for i in range(1, n_reps + 1):
        cache_path = (Path(COMPOSITE_ROOT) / case['comp_expt'] / case['comp_rt']
                      / f'rep_{i:02d}' / 'raw_flux_profile_cache.nc')
        if not cache_path.exists():
            continue
        ds_c = xr.open_dataset(str(cache_path))
        if 'cloud_frac' not in ds_c or 't_hours' not in ds_c:
            ds_c.close()
            continue
        t_hrs  = ds_c['t_hours'].values
        mask_t = (t_hrs >= lst_min) & (t_hrs <= lst_max)
        if mask_t.sum() > 0:
            fracs.append(float(np.nanmean(ds_c['cloud_frac'].values[mask_t])))
        ds_c.close()
    return np.array(fracs)


# ── Load ──────────────────────────────────────────────────────────────────────
abs_profiles = {}
for case in CASES:
    print(f"{RT_LABEL[case['rt']]} ... ", end="", flush=True)
    abs_profiles[case['key']] = load_absolute_profiles(case)
    print(f'{abs_profiles[case["key"]]["n_reps"]} reps')

cloud_fracs = {c['key']: load_cloud_frac(c) for c in CASES}
print('\nCloud-root area fraction  (mean ± 1σ)')
for key, fracs in cloud_fracs.items():
    if len(fracs) > 0:
        mu = fracs.mean()
        sd = fracs.std(ddof=1) if len(fracs) > 1 else 0.0
        print(f'  {key:6s}  α = {mu:.3f} ± {sd:.3f}  ({100*mu:.1f}%)')


## 6. Convective velocity scale $w_*$

Verzijlbergh et al. (2009) domain-integrated convective velocity scale:

$$w_* = \left( c_1 \frac{g}{\theta_0} \int_0^{L_z} \overline{w'\theta_v'}\, dz \right)^{1/3}$$

with $c_1 = 2.5$.  Computed from domain-mean `thv_flux` stats profiles, averaged over **13:00–14:00 LST**.

In [ ]:
# ── w_* timeseries (Verzijlbergh et al. 2009) ───────────────────────────────
# w_*(t) = (c_1 * g / θ_0 * ∫ w'θ_v' dz)^(1/3)
# Computed at every stats timestep; ensemble mean ± between-rep std.
import netCDF4 as _nc4

C1 = 2.5
G  = 9.81

def compute_wstar_timeseries(run_dir):
    """Return (t_sec, w_star_series) for one rep.  NaN where integral ≤ 0."""
    stats_path = sorted(Path(run_dir).glob('cass.default.*.nc'))
    if not stats_path:
        return None, None
    ds = _nc4.Dataset(stats_path[0])
    t_sim  = np.asarray(ds.variables['time'][:])
    zh     = np.asarray(ds.variables['zh'][:])
    thvref = np.asarray(ds.groups['thermo'].variables['thvref'][:])
    flux   = np.asarray(ds.groups['thermo'].variables['thv_flux'][:])  # (time, zh)
    ds.close()

    integral = np.trapezoid(flux, zh, axis=1)    # K m²/s
    theta_0  = float(thvref.mean())
    wstar    = np.where(integral > 0,
                        (C1 * G / theta_0 * np.maximum(integral, 0.0))**(1./3.),
                        np.nan)
    return t_sim, wstar.astype(np.float32)


# ── compute for all reps × both RT; ensemble stack ──────────────────────────
wstar_ens = {}
for case in CASES:
    rt_key = case['key']
    run_root = Path(rs.root) / case['rt']
    t_ref, stack = None, []
    for i in range(1, N_REPS + 1):
        rep_dir = run_root / f'rep_{i:02d}'
        if not rep_dir.exists():
            continue
        t_sec, ws = compute_wstar_timeseries(rep_dir)
        if t_ref is None:
            t_ref = t_sec
        # Truncate all reps to shortest time axis
        n = min(len(t_ref), len(t_sec))
        if len(t_sec) < len(t_ref):
            t_ref = t_ref[:n]
        stack.append(ws[:n])
    # Align all stacked reps to n
    n_min = min(len(s) for s in stack)
    t_ref = t_ref[:n_min]
    stack = np.stack([s[:n_min] for s in stack], axis=0)
    wstar_ens[rt_key] = dict(t_sec=t_ref, stack=stack,
                             mean=np.nanmean(stack, axis=0),
                             std =np.nanstd(stack, axis=0, ddof=0))
    print(f'{rt_key}: n_reps={stack.shape[0]}, nt={stack.shape[1]}, '
          f'peak w* = {np.nanmax(np.nanmean(stack, axis=0)):.2f} m/s')

# ── Figure ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
for rt_key, case in zip(('1D', '3D'), CASES):
    e = wstar_ens[rt_key]
    lst = e['t_sec'] / 3600.0 + LST_OFFSET
    c   = RT_STYLE[case['rt']]['color']
    ax.plot(lst, e['mean'], color=c, lw=1.8, label=RT_LABEL[case['rt']])
    ax.fill_between(lst, e['mean'] - e['std'], e['mean'] + e['std'],
                    color=c, alpha=0.2)
ax.set_xlabel('LST (h)')
ax.set_ylabel(r'$w_*$  (m s$^{-1}$)')
ax.set_xlim(10, 18)
ax.set_xticks(np.arange(10, 18.1, 1))
ax.legend(fontsize=9, frameon=False)
ax.axhline(0, color='0.5', lw=0.6, ls=':')
ax.set_title(fr'{EXPT_KEY}: convective velocity scale $w_*(t)$  '
             fr'(ensemble mean $\pm 1\sigma$, $n$={N_REPS})', fontsize=10)
plt.tight_layout()
if SAVE_PDF:
    plt.savefig(f'wstar_ts_{EXPT_KEY}.pdf', bbox_inches='tight')
plt.show()

# ── LST window summary (unchanged format, but over more windows) ────────────
print('\nw_* averaged over LST windows (ensemble mean ± between-rep std):')
print(f'{"window":<14}{"1D":>12}{"3D":>12}{"Δ":>10}')
for lo, hi in [(11, 12), (12, 13), (13, 14), (14, 15), (15, 16), (16, 17)]:
    row = {}
    for rt_key, case in zip(('1D', '3D'), CASES):
        e = wstar_ens[rt_key]
        lst = e['t_sec'] / 3600.0 + LST_OFFSET
        m = (lst >= lo) & (lst < hi)
        per_rep = np.nanmean(e['stack'][:, m], axis=1)
        row[rt_key] = (np.nanmean(per_rep), np.nanstd(per_rep, ddof=0))
    d = row['3D'][0] - row['1D'][0]
    print(f'{lo:2d}–{hi:2d} LST    '
          f'{row["1D"][0]:6.3f} ± {row["1D"][1]:5.3f}  '
          f'{row["3D"][0]:6.3f} ± {row["3D"][1]:5.3f}  '
          f'{d:+7.3f}')


## 7. LWP integral time scale  $T_\tau$

Domain-mean sliding-window integral time scale of LWP (Taylor 1921, Stull 1988):

$$T_\tau(t) = \Delta t \sum_{\ell=0}^{\ell_0 - 1} \rho_\ell$$

where $\ell_0$ is the first lag where $\rho \leq 0$, estimated over a symmetric 2-hour window centred on $t$.

Cache built by `analysis/t_scale/compute_T_scale.py` (submit via `submit_T_scale.sh`).

In [ ]:
# ── Load T_scale cache ────────────────────────────────────────────────────────
_TS_ROOT = CASS_ROOT / 'analysis' / 'timescale' / EXPT_KEY

_RT_CFG = [
    ('2stream',   RT_STYLE['2stream']['color'], RT_LABEL['2stream']),
    ('raytracer', RT_STYLE['raytracer']['color'], RT_LABEL['raytracer']),
]

ts_data = {}
for rt, color, label in _RT_CFG:
    reps = []
    for i in range(1, N_REPS + 1):
        nc = _TS_ROOT / rt / f'rep_{i:02d}' / 'T_scale.nc'
        if not nc.exists():
            print(f'  MISSING: {nc}')
            continue
        ds_ts = xr.open_dataset(nc)
        reps.append({
            't_lst_h':      ds_ts.t_lst_h.values,
            'T_scale_mean': ds_ts.T_scale_mean.values,
        })
        ds_ts.close()
    ts_data[rt] = reps
    print(f'{rt}: {len(reps)} rep(s) loaded')

# ── Figure 7: domain-mean T_τ — 1D vs 3D ─────────────────────────────────────
LST_PLOT_MIN, LST_PLOT_MAX = 10.0, 16.5   # adjust as needed

fig, ax = plt.subplots(figsize=(10, 4))

for rt, color, label in _RT_CFG:
    reps = ts_data.get(rt, [])
    if not reps:
        continue
    t_h   = reps[0]['t_lst_h']
    stack = np.stack([r['T_scale_mean'] for r in reps])   # (n_reps, T)
    mu    = np.nanmean(stack, axis=0) / 60.0              # seconds → minutes
    sd    = np.nanstd(stack,  axis=0, ddof=0) / 60.0

    ax.plot(t_h, mu, color=color, lw=2.0, label=label)
    ax.fill_between(t_h, mu - sd, mu + sd, color=color, alpha=0.2)

ax.set_xlabel('LST (h)')
ax.set_ylabel(r'$T_\tau$  (min)')
ax.set_xlim(LST_PLOT_MIN, LST_PLOT_MAX)
ax.set_xticks(np.arange(LST_PLOT_MIN, LST_PLOT_MAX + 0.5, 1))
ax.axhline(0, color='gray', lw=0.6, ls='--')
ax.legend(fontsize=9, frameon=False)
# ax.set_title(
#     fr'{EXPT_KEY}: domain-mean LWP integral time scale'
#     r'  (window = 2 h,  nlags = 30 min,  ensemble mean $\pm1\sigma$)',
#     fontsize=10,
# )
plt.tight_layout()
if SAVE_PDF:
    plt.savefig(f'T_scale_{EXPT_KEY}.pdf', bbox_inches='tight')
    print(f'Saved T_scale_{EXPT_KEY}.pdf')
plt.show()

## 8. Lateral entrainment rate ε(z)

Per-updraft fractional entrainment rate from the quasi-steady plume equation
(Kuang & Bretherton 2006; Gentine et al. 2016):

$$\varepsilon(z) = -\frac{1}{h_u(z) - h_\mathrm{env}(z)}\,\frac{d h_u}{d z}$$

with frozen MSE $h = c_p\,\theta_l\,\Pi + g z + L_v q_t$ (warm-only, $q_i = 0$).
Updrafts: $q_l > 10^{-5}$ kg/kg AND $w > 1$ m/s, 3-D connected components, ≥8 cells,
≥3 vertical levels.  Environment: clear columns with a 2-cell subsiding shell excluded.

Cache built by `analysis/entrainment/compute_entrainment.py` (submit via
`submit_entrainment.sh`).  LST 11–17, hourly 3-D dumps, 4 reps × both RT.


In [ ]:
# ── Per-rep M-weighted mean in z_nd; show ensemble mean ± between-rep SEM ──
_ENTR_ROOT = CASS_ROOT / 'analysis' / 'entrainment' / EXPT_KEY
Z_ND_EDGES = np.linspace(0.0, 1.0, 11)
Z_ND_CENT  = 0.5 * (Z_ND_EDGES[:-1] + Z_ND_EDGES[1:])

def _per_rep_wmean_znd(rt, keys=('epsilon',)):
    """Return dict[key] → (n_reps, n_bins) M-weighted means in z_nd bins."""
    out = {k: [] for k in keys}
    for i in range(1, N_REPS + 1):
        f = _ENTR_ROOT / rt / f'rep_{i:02d}' / 'entrainment.nc'
        if not f.exists():
            continue
        with xr.open_dataset(f) as ds:
            pid = ds['plume_id'].values
            cb  = ds['plume_cb_z'].values[pid]
            ct  = ds['plume_ct_z'].values[pid]
            depth = np.where(ct - cb > 100.0, ct - cb, np.nan)
            z_nd = (ds['z'].values - cb) / depth
            M = ds['M'].values
            w = np.where(np.isfinite(M) & (M > 0), M, 0.0)
            for key in keys:
                y = ds[key].values
                means = np.full(len(Z_ND_CENT), np.nan)
                for j in range(len(Z_ND_CENT)):
                    sel = (z_nd >= Z_ND_EDGES[j]) & (z_nd < Z_ND_EDGES[j+1]) \
                          & np.isfinite(y) & (w > 0)
                    if sel.sum() < 20:
                        continue
                    means[j] = np.sum(w[sel] * y[sel]) / np.sum(w[sel])
                out[key].append(means)
    return {k: np.stack(v) for k, v in out.items()}

R_eps = {rt: _per_rep_wmean_znd(rt, keys=('epsilon',)) for rt in ('2stream', 'raytracer')}

# ── Figure: ε(ẑ) mean ± between-rep SEM ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 6))
for rt in ('2stream', 'raytracer'):
    arr = R_eps[rt]['epsilon'] * 1e3   # (n_reps, n_bins), /km
    mean = np.nanmean(arr, axis=0)
    sem  = np.nanstd(arr, axis=0, ddof=1) / np.sqrt(np.sum(np.isfinite(arr), axis=0))
    ok = np.isfinite(mean)
    c = RT_STYLE[rt]['color']
    ax.plot(mean[ok], Z_ND_CENT[ok], color=c, lw=2.0, label=RT_LABEL[rt])
    ax.fill_betweenx(Z_ND_CENT[ok], (mean-sem)[ok], (mean+sem)[ok],
                     color=c, alpha=0.3)
ax.axvline(0, color='0.5', lw=0.7, ls=':')
ax.set_xlabel(r'$\varepsilon$  (km$^{-1}$)')
ax.set_ylabel(r'$\hat z = (z - z_{cb}) / (z_{ct} - z_{cb})$')
ax.set_xlim(-0.5, 3.0)
ax.set_ylim(0, 1)
ax.legend(fontsize=9, frameon=False, loc='lower right')
ax.set_title(fr'{EXPT_KEY}: $\varepsilon(\hat z)$  (M-weighted mean $\pm$ between-rep SEM, n={N_REPS})',
             fontsize=10)
plt.tight_layout()
if SAVE_PDF:
    plt.savefig(f'entrainment_{EXPT_KEY}.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── ε(ẑ) split by LST window (time evolution, non-dimensional) ──────────────

def _load_rt_samples(rt):
    z_nd, eps, M, lst = [], [], [], []
    for i in range(1, N_REPS + 1):
        f = _ENTR_ROOT / rt / f'rep_{i:02d}' / 'entrainment.nc'
        if not f.exists():
            continue
        with xr.open_dataset(f) as ds:
            pid = ds['plume_id'].values
            cb  = ds['plume_cb_z'].values[pid]
            ct  = ds['plume_ct_z'].values[pid]
            depth = np.where(ct - cb > 100.0, ct - cb, np.nan)
            z_nd.append((ds['z'].values - cb) / depth)
            eps.append(ds['epsilon'].values)
            M.append(ds['M'].values)
            lst.append(ds['plume_lst'].values[pid])
    return dict(z_nd=np.concatenate(z_nd), eps=np.concatenate(eps),
                M=np.concatenate(M), lst=np.concatenate(lst))

samples = {rt: _load_rt_samples(rt) for rt in ('2stream', 'raytracer')}

LST_BINS = [(11.0, 13.0), (13.0, 15.0), (15.0, 17.0)]
LST_LBL  = ['LST 11–13', 'LST 13–15', 'LST 15–17']

def _binned_wmean_znd(z_nd, y, M):
    out = np.full(len(Z_ND_CENT), np.nan)
    w = np.where(np.isfinite(M) & (M > 0), M, 0.0)
    for i in range(len(Z_ND_CENT)):
        sel = (z_nd >= Z_ND_EDGES[i]) & (z_nd < Z_ND_EDGES[i+1]) \
              & np.isfinite(y) & (w > 0)
        if sel.sum() < 20:
            continue
        out[i] = np.sum(w[sel] * y[sel]) / np.sum(w[sel])
    return out

fig, axes = plt.subplots(1, 3, figsize=(12, 5), sharey=True, sharex=True)
for ax, (lo, hi), lbl in zip(axes, LST_BINS, LST_LBL):
    for rt in ('2stream', 'raytracer'):
        s = samples[rt]
        sel = (s['lst'] >= lo) & (s['lst'] < hi + 0.5)
        mean = _binned_wmean_znd(s['z_nd'][sel], s['eps'][sel], s['M'][sel])
        ok = np.isfinite(mean)
        ax.plot(mean[ok] * 1e3, Z_ND_CENT[ok],
                color=RT_STYLE[rt]['color'], lw=2.0, label=RT_LABEL[rt])
    ax.axvline(0, color='0.5', lw=0.7, ls=':')
    ax.set_xlabel(r'$\varepsilon$  (km$^{-1}$)')
    ax.set_title(lbl)

axes[0].set_ylabel(r'$\hat z$')
axes[0].legend(fontsize=8, frameon=False, loc='upper right')
axes[0].set_xlim(-0.5, 2.0)
axes[0].set_ylim(0, 1)
fig.suptitle(fr'{EXPT_KEY}: $\varepsilon(\hat z)$ by LST window (M-weighted mean)',
             fontsize=11)
plt.tight_layout()
if SAVE_PDF:
    plt.savefig(f'entrainment_time_{EXPT_KEY}.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── ε and δ vs z  (dimensional, LST 14–16) ──────────────────────────────────
# M-weighted mean ± between-rep SEM, K&B 2006 / Gentine 2016 methodology.
LST_LO_22, LST_HI_22 = 14.0, 16.0
Z_EDGES_22 = np.arange(1200.0, 3001.0, 100.0)
Z_CENT_22  = 0.5 * (Z_EDGES_22[:-1] + Z_EDGES_22[1:])

def _per_rep_wmean_z(rt, keys):
    out = {k: [] for k in keys}
    for i in range(1, N_REPS + 1):
        f = _ENTR_ROOT / rt / f'rep_{i:02d}' / 'entrainment.nc'
        if not f.exists():
            continue
        with xr.open_dataset(f) as ds:
            pid   = ds['plume_id'].values
            lst_s = ds['plume_lst'].values[pid]
            z     = ds['z'].values
            M     = ds['M'].values
            keep = ((lst_s >= LST_LO_22) & (lst_s <= LST_HI_22)
                    & np.isfinite(z) & np.isfinite(M) & (M > 0))
            for key in keys:
                y = ds[key].values
                w = np.where(keep & np.isfinite(y), M, 0.0)
                means = np.full(len(Z_CENT_22), np.nan)
                for j in range(len(Z_CENT_22)):
                    sel = (z >= Z_EDGES_22[j]) & (z < Z_EDGES_22[j+1]) & (w > 0)
                    if sel.sum() < 20:
                        continue
                    means[j] = np.sum(w[sel] * y[sel]) / np.sum(w[sel])
                out[key].append(means)
    return {k: np.stack(v) if v else np.array([]) for k, v in out.items()}

R_ed = {rt: _per_rep_wmean_z(rt, ('epsilon', 'delta')) for rt in ('2stream', 'raytracer')}

fig, axes = plt.subplots(1, 2, figsize=(10, 5.5), sharey=True)
panels = [
    ('epsilon', r'$\varepsilon$  (km$^{-1}$)', 1e3, (-0.5, 3.0), 'Entrainment'),
    ('delta',   r'$\delta$  (km$^{-1}$)',      1e3, (-1, 10),    'Detrainment'),
]
for ax, (key, xlbl, scale, xlim, title) in zip(axes, panels):
    for rt in ('2stream', 'raytracer'):
        arr = R_ed[rt][key] * scale
        if arr.size == 0:
            continue
        mean = np.nanmean(arr, axis=0)
        sem  = np.nanstd(arr, axis=0, ddof=1) / np.sqrt(np.sum(np.isfinite(arr), axis=0))
        ok = np.isfinite(mean)
        c = RT_STYLE[rt]['color']
        ax.plot(mean[ok], Z_CENT_22[ok], color=c, lw=2.0, label=RT_LABEL[rt])
        ax.fill_betweenx(Z_CENT_22[ok], (mean-sem)[ok], (mean+sem)[ok], color=c, alpha=0.3)
    ax.axvline(0, color='0.5', lw=0.7, ls=':')
    ax.set_xlabel(xlbl)
    ax.set_xlim(*xlim)
    ax.set_title(title)
axes[0].set_ylabel('z  (m)')
axes[0].set_ylim(Z_EDGES_22[0], Z_EDGES_22[-1])
axes[0].legend(fontsize=9, frameon=False, loc='upper right')
fig.suptitle(fr'{EXPT_KEY}: LST {LST_LO_22:.0f}–{LST_HI_22:.0f}, M-weighted mean $\pm$ between-rep SEM (n={N_REPS})',
             fontsize=11)
plt.tight_layout()
if SAVE_PDF:
    plt.savefig(f'eps_delta_z_{EXPT_KEY}.pdf', bbox_inches='tight')
plt.show()

print('Paired per-rep differences (3D − 1D); |t|>3.18 ≈ p<0.05, df=3')
for key, xlbl, scale, _, title in panels:
    a1 = R_ed['2stream'][key]   * scale
    a3 = R_ed['raytracer'][key] * scale
    if a1.size == 0 or a3.size == 0:
        continue
    print(f'\n  {title}:')
    print(f'  {"z (m)":>6} {"1D":>8} {"3D":>8} {"Δ":>8} {"t":>6}')
    for j in range(len(Z_CENT_22)):
        d = a3[:, j] - a1[:, j]
        d = d[np.isfinite(d)]
        if len(d) < 2:
            continue
        m = d.mean(); s = d.std(ddof=1) / np.sqrt(len(d))
        t = m / s if s > 0 else np.nan
        print(f'  {Z_CENT_22[j]:>6.0f} {np.nanmean(a1[:,j]):>8.3f} '
              f'{np.nanmean(a3[:,j]):>8.3f} {m:>+8.3f} {t:>+6.2f}')


In [ ]:
# ── Plume buoyancy decomposition, LST 13–15, full vertical profile ──────────
#   Δθ_v ≈ Δθ  +  ε_v · θ̄ · Δq_v  −  θ̄ · q_l,u
#   (dry)     (vapor loading)      (condensate drag)
# θ̄ is ambient θ_env.  Plume thermo fields are averaged over the cloudy
# core only (K&B 2006 / Gentine 2016 methodology); A/M use full plume extent.
EPS_V = 0.608
LST_LO_T, LST_HI_T = 13.0, 15.0
Z_EDGES_T = np.arange(0.0, 3001.0, 100.0)
Z_CENT_T  = 0.5 * (Z_EDGES_T[:-1] + Z_EDGES_T[1:])

def _per_rep_decomp(rt):
    per_rep = {k: [] for k in ('d_theta', 'd_vapor', 'd_cond', 'd_thv')}
    for i in range(1, N_REPS + 1):
        f = _ENTR_ROOT / rt / f'rep_{i:02d}' / 'entrainment.nc'
        if not f.exists():
            continue
        with xr.open_dataset(f) as ds:
            pid = ds['plume_id'].values
            lst_s     = ds['plume_lst'].values[pid]
            z         = ds['z'].values
            M         = ds['M'].values
            theta_u   = ds['theta_u'].values
            theta_env = ds['theta_env'].values
            qt_u      = ds['qt_u'].values
            qt_env    = ds['qt_env'].values
            ql_u      = ds['ql_u'].values
            thv_u     = ds['thv_u'].values
            thv_env   = ds['thv_env'].values

        q_v_u   = qt_u - ql_u
        q_v_env = qt_env                       # env is clear, q_l_env ≈ 0
        d_theta = theta_u - theta_env
        d_vapor = EPS_V * theta_env * (q_v_u - q_v_env)
        d_cond  = -theta_env * ql_u
        d_thv   = thv_u - thv_env

        keep = ((lst_s >= LST_LO_T) & (lst_s <= LST_HI_T)
                & np.isfinite(M) & (M > 0) & np.isfinite(z))
        for key, y in (('d_theta', d_theta), ('d_vapor', d_vapor),
                       ('d_cond',  d_cond),  ('d_thv',   d_thv)):
            means = np.full(len(Z_CENT_T), np.nan)
            w = np.where(keep & np.isfinite(y), M, 0.0)
            for j in range(len(Z_CENT_T)):
                sel = (z >= Z_EDGES_T[j]) & (z < Z_EDGES_T[j+1]) & (w > 0)
                if sel.sum() < 20:
                    continue
                means[j] = np.sum(w[sel] * y[sel]) / np.sum(w[sel])
            per_rep[key].append(means)
    return {k: np.stack(v) for k, v in per_rep.items()}

R_th = {rt: _per_rep_decomp(rt) for rt in ('2stream', 'raytracer')}

fig, axes = plt.subplots(1, 4, figsize=(16, 6), sharey=True)
panels = [
    ('d_theta', r'$\Delta\theta$  (K)',                     'Dry $\theta$ excess',   (-1.0, 0.5)),
    ('d_vapor', r'$\epsilon_v\,\theta\,\Delta q_v$  (K)',   'Vapor loading',         (-0.2, 2.0)),
    ('d_cond',  r'$-\theta\,q_{l,u}$  (K)',                 'Condensate drag',       (-1.0, 0.1)),
    ('d_thv',   r'$\Delta\theta_v$  (K)',                   'Total buoyancy excess', (-0.5, 0.5)),
]
for ax, (key, xlbl, title, xlim) in zip(axes, panels):
    for rt in ('2stream', 'raytracer'):
        arr  = R_th[rt][key]
        mean = np.nanmean(arr, axis=0)
        sem  = np.nanstd(arr, axis=0, ddof=1) / np.sqrt(np.sum(np.isfinite(arr), axis=0))
        ok   = np.isfinite(mean)
        c    = RT_STYLE[rt]['color']
        ax.plot(mean[ok], Z_CENT_T[ok], color=c, lw=2.0, label=RT_LABEL[rt])
        ax.fill_betweenx(Z_CENT_T[ok], (mean-sem)[ok], (mean+sem)[ok],
                         color=c, alpha=0.3)
    ax.axvline(0, color='0.5', lw=0.7, ls=':')
    ax.set_xlabel(xlbl)
    ax.set_title(title, fontsize=10)
    ax.set_xlim(*xlim)
axes[0].set_ylabel('z  (m)')
axes[0].set_ylim(Z_EDGES_T[0], Z_EDGES_T[-1])
axes[0].legend(fontsize=9, frameon=False, loc='upper right')
fig.suptitle(fr'{EXPT_KEY}: buoyancy decomposition (LST {LST_LO_T:.0f}–{LST_HI_T:.0f}, '
             fr'M-weighted mean $\pm$ between-rep SEM, $n$={N_REPS})', fontsize=11)
plt.tight_layout()
if SAVE_PDF:
    plt.savefig(f'plume_buoyancy_decomp_{EXPT_KEY}.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── LNB (Level of Neutral Buoyancy) over LST ──────────────────────────────
# Per snapshot × rep: find the highest z-bin where the M-weighted mean
# Δθ_v = θ_v,u − θ_v,env is positive. That is the empirical LNB for that
# snapshot. Plot ensemble mean ± between-rep std vs LST.

Z_EDGES_LNB = np.arange(1200.0, 3501.0, 100.0)
Z_CENT_LNB  = 0.5 * (Z_EDGES_LNB[:-1] + Z_EDGES_LNB[1:])

def _lnb_per_rep(rt):
    per_rep_lst, per_rep_lnb = [], []
    for i in range(1, N_REPS + 1):
        f = _ENTR_ROOT / rt / f'rep_{i:02d}' / 'entrainment.nc'
        if not f.exists():
            continue
        with xr.open_dataset(f) as ds:
            pid     = ds['plume_id'].values
            snap_id = ds['snap_id'].values
            z       = ds['z'].values
            M       = ds['M'].values
            dthv    = ds['thv_u'].values - ds['thv_env'].values
            lst_all = ds['plume_lst'].values[pid]
        lst_vals, lnb_vals = [], []
        for s in np.unique(snap_id):
            sel = snap_id == s
            if sel.sum() == 0:
                continue
            z_s = z[sel]; M_s = M[sel]; dthv_s = dthv[sel]
            w = np.where(np.isfinite(M_s) & (M_s > 0) & np.isfinite(dthv_s), M_s, 0.0)
            mean_dthv = np.full(len(Z_CENT_LNB), np.nan)
            for j in range(len(Z_CENT_LNB)):
                bin_sel = (z_s >= Z_EDGES_LNB[j]) & (z_s < Z_EDGES_LNB[j+1]) & (w > 0)
                if bin_sel.sum() < 10:
                    continue
                mean_dthv[j] = np.sum(w[bin_sel] * dthv_s[bin_sel]) / np.sum(w[bin_sel])
            # LNB = highest z-centre where Δθ_v > 0 (buoyant)
            buoyant = np.isfinite(mean_dthv) & (mean_dthv > 0)
            if not buoyant.any():
                continue
            lnb = Z_CENT_LNB[np.where(buoyant)[0][-1]]
            lst_vals.append(lst_all[sel][0])
            lnb_vals.append(lnb)
        per_rep_lst.append(np.asarray(lst_vals))
        per_rep_lnb.append(np.asarray(lnb_vals))
    return per_rep_lst, per_rep_lnb

lst_1d, lnb_1d = _lnb_per_rep('2stream')
lst_3d, lnb_3d = _lnb_per_rep('raytracer')

# Ensemble aggregation: all reps share the same hourly snapshot LSTs
def _ens(lst_list, lnb_list):
    stack = np.stack([np.interp(lst_list[0], li, li_lnb)
                       for li, li_lnb in zip(lst_list, lnb_list)])
    return lst_list[0], np.nanmean(stack, axis=0), np.nanstd(stack, axis=0, ddof=0)

t_1d, mu_1d, sd_1d = _ens(lst_1d, lnb_1d)
t_3d, mu_3d, sd_3d = _ens(lst_3d, lnb_3d)

fig, ax = plt.subplots(figsize=(9, 4))
for t, mu, sd, rt in [(t_1d, mu_1d, sd_1d, '2stream'),
                      (t_3d, mu_3d, sd_3d, 'raytracer')]:
    c = RT_STYLE[rt]['color']
    ax.plot(t, mu, color=c, lw=2.0, marker='o', markersize=5, label=RT_LABEL[rt])
    ax.fill_between(t, mu - sd, mu + sd, color=c, alpha=0.3)
ax.set_xlabel('LST  (h)')
ax.set_ylabel('LNB  (m)')
ax.set_xlim(min(np.nanmin(t_1d), np.nanmin(t_3d)) - 0.3,
            max(np.nanmax(t_1d), np.nanmax(t_3d)) + 0.3)
ax.legend(fontsize=9, frameon=False)
ax.set_title(fr'{EXPT_KEY}: level of neutral buoyancy over time  '
             fr'(highest z where M-weighted $\Delta\theta_v > 0$, mean $\pm 1\sigma$, $n$={N_REPS})',
             fontsize=10)
plt.tight_layout()
if SAVE_PDF:
    plt.savefig(f'lnb_over_time_{EXPT_KEY}.pdf', bbox_inches='tight')
plt.show()

# Table of LNB differences
print('LNB (m)  mean ± std across reps')
print(f'{"LST":>6}  {"1D":>12}  {"3D":>12}  {"3D − 1D":>10}')
n = min(len(t_1d), len(t_3d))
for k in range(n):
    d = mu_3d[k] - mu_1d[k]
    print(f'{t_1d[k]:6.1f}  {mu_1d[k]:6.0f} ± {sd_1d[k]:4.0f}  '
          f'{mu_3d[k]:6.0f} ± {sd_3d[k]:4.0f}  {d:+10.0f}')


In [ ]:
# ── Gentine-style joint distributions, LST 14–16 ────────────────────────────
#   Row 1: (ε, z) on LOG ε axis, Gentine-style log-spaced bins
#   Row 2: (MSE, z) in 0.1 K bins
#   Cols: 1D, 3D, 3D−1D
# Mass flux normalization: sum(M) / (A_domain × N_snap × N_reps)
# Black below 1e-4 kg m⁻² s⁻¹ bin⁻¹ (Gentine 2016 cutoff).

CP = 1005.0
LST_LO, LST_HI = 14.0, 16.0

# Log-spaced ε edges: 10^(-x) m^-1 with x ∈ [2.3, 5.0] in 0.05 steps → 0.01 … 5 /km
_eps_x   = np.arange(2.3, 5.05, 0.05)
EPS_EDGES = np.sort(10 ** (-_eps_x) * 1e3)     # /km, monotone increasing
MSE_EDGES = np.arange(325.0, 345.01, 0.1)      # 0.1 K bins
Z_EDGES   = np.arange(1200.0, 3001.0, 50.0)    # 50 m bins

MF_LOW_CUT = 1e-4                              # Gentine 2016 threshold

def _mf_hist(rt, xkey, xscale, xedges, lst_lo=LST_LO, lst_hi=LST_HI):
    hist_total = np.zeros((len(xedges)-1, len(Z_EDGES)-1), dtype=np.float64)
    n_snap_reps = 0
    A_dom = None
    for i in range(1, N_REPS + 1):
        rep_dir = Path(str(rs.root)) / rt / f'rep_{i:02d}'
        f = _ENTR_ROOT / rt / f'rep_{i:02d}' / 'entrainment.nc'
        if not f.exists():
            continue
        with xr.open_dataset(rep_dir / 'thl.nc', decode_times=False) as dsx:
            t = dsx['time'].values
            lst = t / 3600.0 + LST_OFFSET
            n_snap_this = int(((lst >= lst_lo) & (lst <= lst_hi)).sum())
            if A_dom is None:
                x = dsx['x'].values; y = dsx['y'].values
                A_dom = (x[-1] - x[0]) * (y[-1] - y[0])
        with xr.open_dataset(f) as ds:
            pid   = ds['plume_id'].values
            lst_s = ds['plume_lst'].values[pid]
            z     = ds['z'].values
            xv    = ds[xkey].values * xscale
            M     = ds['M'].values
        keep = (np.isfinite(z) & np.isfinite(xv) & np.isfinite(M) & (M > 0)
                & (lst_s >= lst_lo) & (lst_s <= lst_hi))
        H, _, _ = np.histogram2d(
            xv[keep], z[keep],
            bins=[xedges, Z_EDGES],
            weights=M[keep],
        )
        hist_total += H
        n_snap_reps += n_snap_this
    return (hist_total / (A_dom * n_snap_reps)
            if n_snap_reps else hist_total)

mf_eps_1d = _mf_hist('2stream',   'epsilon', 1e3,     EPS_EDGES)
mf_eps_3d = _mf_hist('raytracer', 'epsilon', 1e3,     EPS_EDGES)
mf_mse_1d = _mf_hist('2stream',   'h_u',     1.0/CP,  MSE_EDGES)
mf_mse_3d = _mf_hist('raytracer', 'h_u',     1.0/CP,  MSE_EDGES)

print(f'LST {LST_LO:.0f}–{LST_HI:.0f}')
print(f'ε  peak: 1D={mf_eps_1d.max():.2e}, 3D={mf_eps_3d.max():.2e}')
print(f'MSE peak: 1D={mf_mse_1d.max():.2e}, 3D={mf_mse_3d.max():.2e}')

import matplotlib as _mpl
_VIR_CUT = _mpl.cm.viridis.copy()
_VIR_CUT.set_under('black')

fig, axes = plt.subplots(2, 3, figsize=(14, 9), sharey=True)
for row, (arr1, arr3, xedges, xlabel, xlog) in enumerate([
    (mf_eps_1d, mf_eps_3d, EPS_EDGES, r'$\varepsilon$  (km$^{-1}$)',      True),
    (mf_mse_1d, mf_mse_3d, MSE_EDGES, r'MSE $= h_u / c_p$  (K)',          False),
]):
    diff  = arr3 - arr1
    vmax  = np.percentile(np.maximum(arr1, arr3).ravel(), 99)
    vdiff = np.percentile(np.abs(diff).ravel(), 99)
    panels = [
        (arr1.T,  '1D (2stream)',   MF_LOW_CUT, vmax,  _VIR_CUT),
        (arr3.T,  '3D (raytracer)', MF_LOW_CUT, vmax,  _VIR_CUT),
        (diff.T,  '3D − 1D',       -vdiff,      vdiff, 'RdBu_r'),
    ]
    for col, (arr, title, vlo, vhi, cmap) in enumerate(panels):
        ax = axes[row, col]
        pcm = ax.pcolormesh(xedges, Z_EDGES, arr,
                            vmin=vlo, vmax=vhi, cmap=cmap, shading='auto')
        plt.colorbar(pcm, ax=ax, label=r'kg m$^{-2}$ s$^{-1}$ bin$^{-1}$',
                     extend=('min' if cmap is _VIR_CUT else 'neither'))
        ax.set_xlabel(xlabel)
        if xlog:
            ax.set_xscale('log')
            ax.set_xlim(EPS_EDGES[0], EPS_EDGES[-1])
        if row == 0:
            ax.set_title(title, fontsize=10)
    axes[row, 0].set_ylabel('z  (m)')

fig.suptitle(fr'{EXPT_KEY}: joint mass-flux density, LST {LST_LO:.0f}–{LST_HI:.0f}  '
             fr'(ε log, MSE 0.1 K, z 50 m)',
             fontsize=11)
plt.tight_layout()
if SAVE_PDF:
    plt.savefig(f'entr_joint_hist_{EXPT_KEY}.pdf', bbox_inches='tight')
plt.show()
